# 03 — Model Training

EfficientNet-B4 + TimeDistributed. 9 augmentation/cutout configurations.

**Requires:** Data from notebooks 01–02, GPU.

In [ ]:
import os, random
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.mixed_precision import set_global_policy
import matplotlib.pyplot as plt

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

policy = "mixed_float16" if tf.config.list_physical_devices("GPU") else "float32"
set_global_policy(policy)
tf.keras.backend.clear_session()
print(f"TF {tf.__version__}, policy: {policy}")

## Experiment Configurations

Nine configs varying augmentation and cutout. `onlymoreaug` uses `high_prob=True` in `create_augmentations()`.

In [ ]:
CONFIGS = {
    "baseline":            dict(use_augmentations=False, use_cutouts=False, fill_fake="zero",   fill_star="zero",   high_prob_aug=False),
    "onlyaug":             dict(use_augmentations=True,  use_cutouts=False, fill_fake="zero",   fill_star="zero",   high_prob_aug=False),
    "onlymoreaug":         dict(use_augmentations=True,  use_cutouts=False, fill_fake="zero",   fill_star="zero",   high_prob_aug=True),
    "randomfilledwithaug": dict(use_augmentations=True,  use_cutouts=True,  fill_fake="random", fill_star="random", high_prob_aug=False),
    "blackfilledwithaug":  dict(use_augmentations=True,  use_cutouts=True,  fill_fake="zero",   fill_star="zero",   high_prob_aug=False),
    "whitefilledwithaug":  dict(use_augmentations=True,  use_cutouts=True,  fill_fake="white",  fill_star="white",  high_prob_aug=False),
    "noaugrandomfill":     dict(use_augmentations=False, use_cutouts=True,  fill_fake="random", fill_star="random", high_prob_aug=False),
    "noaugblackfill":      dict(use_augmentations=False, use_cutouts=True,  fill_fake="zero",   fill_star="zero",   high_prob_aug=False),
    "noaugwhitefill":      dict(use_augmentations=False, use_cutouts=True,  fill_fake="white",  fill_star="white",  high_prob_aug=False),
}

ACTIVE_CONFIG = "blackfilledwithaug"
cfg = CONFIGS[ACTIVE_CONFIG]
print(f"Config: {ACTIVE_CONFIG}"); print(cfg)

## Dataset Class

Each batch yields 4 samples per video pair: original real, original fake, augmented/cutout real, augmented/cutout fake. Polygon selection uses SSIM on the middle frame, landmarks from the real frame.

In [ ]:
# Assumes all functions from notebook 02 are available:
#   extract_frames, compute_ssim_mask, extract_landmarks,
#   generate_candidate_polygons, select_best_polygon,
#   apply_seferbekov_cutout, apply_star_on_real_frames,
#   create_augmentations, resize_and_normalize

class VideoDatasetSeferbekov(tf.keras.utils.Sequence):
    def __init__(self, video_pairs, sequence_length=12, batch_size=4,
                 shuffle=True, use_augmentations=True, use_cutouts=True,
                 fill_type_fake="random", fill_type_star="random",
                 high_prob_aug=False, p_cutout=0.5, rho_thresh=0.3,
                 ssim_bin=0.5, seed=42, verbose=False):

        self.video_pairs = video_pairs
        self.sequence_length = sequence_length
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.use_augmentations = use_augmentations
        self.use_cutouts = use_cutouts
        self.fill_type_fake = fill_type_fake
        self.fill_type_star = fill_type_star
        self.high_prob_aug = high_prob_aug
        self.p_cutout = p_cutout
        self.rho_thresh = rho_thresh
        self.ssim_bin = ssim_bin
        self.verbose = verbose

        random.seed(seed); np.random.seed(seed)
        try: tf.random.set_seed(seed)
        except: pass

        self.indexes = np.arange(len(self.video_pairs))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.video_pairs) / self.batch_size))

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.video_pairs))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __getitem__(self, index):
        idxs = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        return self._generate([self.video_pairs[k] for k in idxs])

    def _get_ssim_polygon(self, real_frames, fake_frames):
        mid = len(real_frames) // 2
        mask = compute_ssim_mask(real_frames[mid], fake_frames[mid])
        lm = extract_landmarks(real_frames[mid])
        if lm is None:
            return None
        polys = generate_candidate_polygons(lm, n_random=6)
        if not polys:
            return None
        return select_best_polygon(polys, mask, min_overlap=self.rho_thresh,
                                   ssim_bin=self.ssim_bin, mode="sim")

    def _generate(self, batch_pairs):
        total = self.batch_size * 4
        X = np.empty((total, self.sequence_length, 224, 224, 3), dtype=np.float32)
        y = np.empty((total, self.sequence_length, 1), dtype=np.float32)

        for i, (real_path, fake_path) in enumerate(batch_pairs):
            try:
                real_raw, fake_raw = extract_frames(
                    real_path, fake_path,
                    frame_step=FRAME_STEP, frame_count=self.sequence_length)

                if self.use_cutouts:
                    # star cutout on real
                    if np.random.rand() < self.p_cutout:
                        real_cut = apply_star_on_real_frames(
                            real_raw, enable=True, prob=1.0,
                            star_fill_type=self.fill_type_star)
                    else:
                        real_cut = [f.copy() for f in real_raw]

                    # polygon cutout on fake
                    if np.random.rand() < self.p_cutout:
                        poly = self._get_ssim_polygon(real_raw, fake_raw)
                        if poly is not None:
                            _, fake_cut = apply_seferbekov_cutout(
                                fake_raw, fake_raw, polygon=poly,
                                fake_fill_type=self.fill_type_fake)
                        else:
                            fake_cut = [f.copy() for f in fake_raw]
                    else:
                        fake_cut = [f.copy() for f in fake_raw]
                else:
                    real_cut = [f.copy() for f in real_raw]
                    fake_cut = [f.copy() for f in fake_raw]

                augs = create_augmentations(high_prob=self.high_prob_aug) \
                       if self.use_augmentations else None

                # original (no aug, no cutout)
                orig_real, orig_fake = [], []
                for frame in real_raw:
                    r, _ = resize_and_normalize(frame, frame, augment=False)
                    orig_real.append(r)
                for frame in fake_raw:
                    f, _ = resize_and_normalize(frame, frame, augment=False)
                    orig_fake.append(f)

                # augmented + cutout
                aug_real, aug_fake = [], []
                for frame in real_cut:
                    r, _ = resize_and_normalize(frame, frame,
                               augment=self.use_augmentations, augment_instance=augs)
                    aug_real.append(r)
                for frame in fake_cut:
                    f, _ = resize_and_normalize(frame, frame,
                               augment=self.use_augmentations, augment_instance=augs)
                    aug_fake.append(f)

                base = 4 * i
                X[base]   = np.array(orig_real);  y[base]   = np.ones((self.sequence_length, 1), dtype=np.float32)
                X[base+1] = np.array(orig_fake);  y[base+1] = np.zeros((self.sequence_length, 1), dtype=np.float32)
                X[base+2] = np.array(aug_real);   y[base+2] = np.ones((self.sequence_length, 1), dtype=np.float32)
                X[base+3] = np.array(aug_fake);   y[base+3] = np.zeros((self.sequence_length, 1), dtype=np.float32)

            except Exception as e:
                print(f"Error pair {i}: {e}")
                base = 4 * i
                for j in range(4):
                    X[base+j] = np.zeros((self.sequence_length, 224, 224, 3), dtype=np.float32)
                    y[base+j] = np.zeros((self.sequence_length, 1), dtype=np.float32)

        return X, y

## Model Architecture

In [ ]:
def create_model(seq_len=12, img_size=224):
    inputs = layers.Input(shape=(seq_len, img_size, img_size, 3))

    backbone = EfficientNetB4(weights="imagenet", include_top=False,
                              input_shape=(img_size, img_size, 3))
    x = layers.TimeDistributed(backbone)(inputs)
    x = layers.TimeDistributed(layers.GlobalAveragePooling2D())(x)
    x = layers.TimeDistributed(layers.Dropout(0.25))(x)
    outputs = layers.TimeDistributed(
        layers.Dense(1, activation="sigmoid",
                     kernel_regularizer=regularizers.l2(1e-3),
                     dtype="float32")
    )(x)

    return models.Model(inputs, outputs)

def video_loss(y_true, y_pred):
    return tf.keras.losses.binary_crossentropy(
        tf.reduce_mean(y_true, axis=1),
        tf.reduce_mean(y_pred, axis=1))

model = create_model()
model.summary()

## Training

In [ ]:
train_dataset = VideoDatasetSeferbekov(
    train_pairs, use_augmentations=cfg["use_augmentations"],
    use_cutouts=cfg["use_cutouts"], fill_type_fake=cfg["fill_fake"],
    fill_type_star=cfg["fill_star"], high_prob_aug=cfg["high_prob_aug"])

val_dataset = VideoDatasetSeferbekov(
    val_pairs, shuffle=False, use_augmentations=False, use_cutouts=False)

test_dataset = VideoDatasetSeferbekov(
    test_pairs, shuffle=False, use_augmentations=False, use_cutouts=False)

cosine_lr = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3, decay_steps=1000, alpha=0.0)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=cosine_lr, clipvalue=1.0),
    loss=video_loss,
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])

early_stop = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True,
    verbose=1, mode="min", min_delta=0.001)

history = model.fit(train_dataset, validation_data=val_dataset,
                    epochs=20, callbacks=[early_stop], verbose=1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key, title in zip(axes, ["loss", "accuracy", "auc"],
                           ["Loss", "Accuracy", "AUC"]):
    ax.plot(history.history[key], label="train")
    ax.plot(history.history[f"val_{key}"], label="val")
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
model.save(f"weights/{ACTIVE_CONFIG}.h5")
print(f"Saved weights/{ACTIVE_CONFIG}.h5")